In [25]:
# SPDX-License-Identifier: CC-BY-4.0
# Code for "Active Continual Learning with Metaplastic Binary Bayesian Neural Networks"
# Kellian Cottart, Théo Ballet, Djohan Bonnet, Damien Querlioz
# Portions of the code are adapted from the Pytorch project (BSD-3-Clause)
# Author: Kellian Cottart <kellian.cottart@gmail.com>
# Date: 2025-30-01

In [26]:
configuration = {
  "network": "binarybayesianmlp",
  "network_params": {
    "layers": [8192, 19],
    "temperature": 1,
    "activation_fn": "reversebinarygate",
    "activation_params": {
      "width": 1
    }
  },
  "optimizer": "bimu",
  "optimizer_params": {
    "likelihood_multiplier": 16.7,
    "kl_multiplier": 0.53,
    "lr_max": 0.065,
    "N": 1600,
    "lr": 48.7
  },
  "task": "openloris",
  "task_params": {
    "unbalanced": {
      "n_min": 0,
      "threshold_max": 700,
      "remove_ratio_min": 0.5,
      "remove_ratio_max": 0.8
    },
    "subfeatures": 8192,
    "dataset_normalisation": "none",
    "feature_extraction": True,
    "feature_extractor": "vgg19",
    "classes": [
      "bottle",
      "bowl",
      "corkscrew",
      "cottonswab",
      "cup",
      "cushion",
      "glasses",
      "knife",
      "ladle",
      "mask",
      "paper_cutter",
      "pencil",
      "plasticbag",
      "plug",
      "pot",
      "scissors",
      "stapler",
      "thermometer",
      "toy"
    ]
  },
  "n_tasks": 12,
  "epochs": 1,
  "seed": 1000,
  "n_train_samples": 10,
  "n_test_samples": 10,
  "train_batch_size": 1,
  "test_batch_size": 4
}

In [27]:
from optimizers import *
from utils import *
from models import *
from datetime import datetime
import os
import json
from shutil import rmtree
import argparse
from copy import deepcopy
import numpy as np
from torch import manual_seed, cat
from time import time
import jax
import jax.numpy as jnp
from jax.numpy import expand_dims
import equinox as eqx


In [ ]:
# Initialize the random number generator
manual_seed(configuration["seed"])
np.random.seed(configuration["seed"])
rng = jax.random.key(configuration["seed"])
# Load the dataset
loader = GPULoading(device="cpu")
task_params = configuration["task_params"] if "task_params" in configuration else {
}
n_splits_per_epoch = configuration.get("n_splits_per_epoch", 1)
train, test, shape, num_classes = loader.task_selection(
    configuration["task"], **task_params)
model_key, rng = jax.random.split(rng)
# Configure the model
model, model_state = configure_networks(configuration, model_key)
optimizer, opt_state = configure_optimizer(
                configuration, eqx.filter(model, eqx.is_array))
# GENERATING A HUGE ARRAY OF KEYS, ASSURING THAT THE KEYS ARE UNIQUE
trkey, tekey, rng = jax.random.split(rng, 3)
training_core_keys = jax.random.split(trkey, 1)


train_samples = configuration["n_train_samples"]
test_samples = configuration["n_test_samples"]

# ---------- Preparing Dataloaders --------------
# train has 12 TensorDatasets in a list. we want to concatenate them into a single TensorDataset, and then create a dataloader from it.

images_list = []
labels_list = []
for task in train:
    images_list.append(task.tensors[0])
    labels_list.append(task.tensors[1])
full_images = cat(images_list, dim=0)
full_labels = cat(labels_list, dim=0)
train = to_dataloader(
    [TensorDataset(full_images, full_labels)], 1, num_classes, fits_in_memory=True)[0]
images = train[0]
labels = train[1]
print(f"Image shape: {images.shape}, Label shape: {labels.shape}")

Full images shape: torch.Size([138029, 8192, 1, 1]), Full labels shape: torch.Size([138029])
Image shape: (138029, 1, 8192, 1, 1), Label shape: (138029, 1, 19)


In [29]:
@eqx.filter_jit
def test_epoch(model, images, labels, state, samples, rng):
    def step_fn(carry, batch):
        state, rng = carry
        image, label = batch
        predictions, new_state = jax.vmap(
            model,
            axis_name="batch",
            in_axes=(0, None, None, None),
            out_axes=(0, None)
        )(image, state, samples, rng)
        output = jax.nn.log_softmax(predictions, axis=-1).mean(axis=1)
        return (new_state, rng), output
    init_carry = (state, rng)
    (state, rng), outputs = jax.lax.scan(
        step_fn,
        init_carry,
        (images, labels)
    )
    return outputs

# Warmup (compile)
outputs = test_epoch(model, images, labels, model_state, test_samples, rng)

# Timing
%timeit -r 10 -n 1 test_epoch(model, images, labels, model_state, test_samples, rng).block_until_ready()

5.96 s ± 16.4 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)


In [30]:
@eqx.filter_jit
def train_epoch(model, opt_state, optimizer, images, labels, samples, rng, init_state=None):
    
    dynamic_init_state, static_state = eqx.partition(model, eqx.is_array)
    def step_fn(carry, batch):
        dynamic_state, opt_state, rng, state = carry
        image, label = batch
        model = eqx.combine(dynamic_state, static_state)
        @eqx.filter_value_and_grad(has_aux=True)
        def bayesian_loss_fn(model, images, labels, samples, rng, init_state=None):
            """ Loss function for Bayesian models. """
            # Same rng for all images in the batch, but different for each sample
            predictions, state = jax.vmap(partial(model, backwards=True),
                                        in_axes=(0, None, None, None), out_axes=(0, None))(images, init_state, samples, rng)
            output = jax.nn.log_softmax(
                predictions, axis=-1).mean(axis=1) * labels 
            loss = -jnp.sum(output, axis=-1).sum()
            return loss, state

        (loss, new_state), grads = bayesian_loss_fn(model, image, label, samples, rng, init_state)
        updates, opt_state = optimizer.update(grads, opt_state, dynamic_state)
        dynamic_state = optax.apply_updates(dynamic_state, updates)
        return (dynamic_state, opt_state, rng, new_state), loss

    # Initialize scan carry
    init_carry = (dynamic_init_state, opt_state, rng, init_state)

    (dynamic_init_state, opt_state, rng, state), losses = jax.lax.scan(
        step_fn,
        init_carry,
        (images, labels)
    )
    model = eqx.combine(dynamic_init_state, static_state)
    return losses

# Warmup (important for JIT)
losses = train_epoch(model, opt_state, optimizer, images, labels, train_samples, rng, init_state=None)

# Timing
%timeit -r 10 -n 1 train_epoch(model, opt_state, optimizer, images, labels, train_samples, rng, init_state=None).block_until_ready()

11.4 s ± 35 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)
